# Projet Data Scientist: Prédiction de la Qualité de l'Air

## Objectif
Prédire la qualité de l'air dans une ville intelligente en utilisant des données météorologiques
et des caractéristiques des localisations.

## Dataset
- **Train.csv**: Données d'entraînement avec variables météorologiques et qualité de l'air
- **Test.csv**: Données de test pour les prédictions
- **airqo_metadata.csv**: Métadonnées des localisations

## Approche
1. Feature engineering sur les séries temporelles météorologiques
2. Intégration des métadonnées des localisations
3. Entraînement de modèles avancés (LightGBM, XGBoost, CatBoost)
4. Dashboard Streamlit pour le déploiement

In [23]:
# Import des bibliothèques nécessaires
import joblib
import warnings

import plotly.express as px


import catboost as cat
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")

In [24]:
# Chargement des données
print("Chargement des données...")

train = pd.read_csv("Train.csv")
test = pd.read_csv("Test.csv")
metadata = pd.read_csv("airqo_metadata.csv")

print(f"Train: {train.shape}")
print(f"Test: {test.shape}")
print(f"Metadata: {metadata.shape}")

Chargement des données...
Train: (15539, 9)
Test: (5035, 8)
Metadata: (5, 17)


In [49]:
# Analyse exploratoire des données
print("Analyse exploratoire des données")

print("Colonnes du train:")
print(train.columns.tolist())

print("\nÉchantillon du train:")
print(train.head())

# Statistiques de la variable cible
print("\nStatistiques de la qualité de l'air (target):")
target_stats = train["target"].describe()
print(target_stats)

# Distribution par localisation
print("\nRépartition des localisations:")
location_counts = train["location"].value_counts()
print(location_counts)

# Métadonnées des localisations
print("\nMétadonnées des localisations:")
print(metadata)

Analyse exploratoire des données
Colonnes du train:
['ID', 'location', 'temp', 'precip', 'rel_humidity', 'wind_dir', 'wind_spd', 'atmos_press', 'target', 'temp_mean', 'temp_std', 'temp_min', 'temp_max', 'precip_mean', 'precip_std', 'precip_min', 'precip_max', 'rel_humidity_mean', 'rel_humidity_std', 'rel_humidity_min', 'rel_humidity_max', 'wind_dir_mean', 'wind_dir_std', 'wind_dir_min', 'wind_dir_max', 'wind_spd_mean', 'wind_spd_std', 'wind_spd_min', 'wind_spd_max', 'atmos_press_mean', 'atmos_press_std', 'atmos_press_min', 'atmos_press_max', 'temp_median', 'temp_skew', 'temp_kurt', 'temp_trend', 'temp_roll_mean_10', 'precip_median', 'precip_skew', 'precip_kurt', 'precip_trend', 'precip_roll_mean_10', 'rel_humidity_median', 'rel_humidity_skew', 'rel_humidity_kurt', 'rel_humidity_trend', 'rel_humidity_roll_mean_10', 'wind_dir_median', 'wind_dir_skew', 'wind_dir_kurt', 'wind_dir_trend', 'wind_dir_roll_mean_10', 'wind_spd_median', 'wind_spd_skew', 'wind_spd_kurt', 'wind_spd_trend', 'wind_s

In [41]:
# Conversion des features météorologiques
def replace_nan(x):
    if x == " ":
        return np.nan
    else:
        return float(x)

features = ["temp", "precip", "rel_humidity", "wind_dir", "wind_spd", "atmos_press"]

print("Conversion des features météorologiques...")


def extract_features(df, features):
    for feature in features:
        values = df[feature].apply(lambda x: [v for v in x if not np.isnan(v)])
        df[f"{feature}_mean"] = df[feature].apply(lambda x: np.nanmean(x))
        df[f"{feature}_std"] = df[feature].apply(lambda x: np.nanstd(x))
        df[f"{feature}_min"] = df[feature].apply(lambda x: np.nanmin(x))
        df[f"{feature}_max"] = df[feature].apply(lambda x: np.nanmax(x))
        df[f"{feature}_median"] = df[feature].apply(lambda x: np.nanmedian(x))
        df[f"{feature}_skew"] = df[feature].apply(lambda x: pd.Series(x).skew())
        df[f"{feature}_kurt"] = df[feature].apply(lambda x: pd.Series(x).kurtosis())
        # Trend: slope of linear regression
        df[f"{feature}_trend"] = df[feature].apply(lambda x: np.polyfit(range(len(x)), x, 1)[0] if len([v for v in x if not np.isnan(v)]) > 1 else 0)
        # Rolling mean of last 10
        df[f"{feature}_roll_mean_10"] = df[feature].apply(lambda x: np.nanmean(x[-10:]) if len(x) >= 10 else np.nanmean(x))
    return df

# Conversion des features météorologiques en listes de floats si nécessaire
for feature in features:
    if isinstance(train[feature].iloc[0], str):
        train[feature] = train[feature].apply(lambda x: [replace_nan(val) for val in x.split(',')])
        test[feature] = test[feature].apply(lambda x: [replace_nan(val) for val in x.split(',')])

train = extract_features(train, features)
test = extract_features(test, features)

# Préparation finale des données
def prepare_final_data(train, test, features, metadata):
    """Prépare les données finales pour l'entraînement"""

    # Suppression des colonnes de listes originales
    data = pd.concat([train, test], sort=False).reset_index(drop=True)
    data.drop(features, axis=1, inplace=True)

    # Merge metadata
    data = data.merge(metadata, on='location', how='left')

    # Fill NaN in metadata with mean
    metadata_cols = [col for col in data.columns if col not in ['ID', 'location', 'target', 'fold']]
    for col in metadata_cols:
        if data[col].dtype in ['float64', 'int64']:
            data[col] = data[col].fillna(data[col].mean())

    # Encode location
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    data['location_encoded'] = le.fit_transform(data['location'])

    # Séparation train/test
    train_final = data[data["target"].notnull()].reset_index(drop=True)
    test_final = data[data["target"].isna()].reset_index(drop=True)

    # Suppression des colonnes inutiles
    cols_to_drop = ["ID", "location", "Unnamed: 0"]
    train_final = train_final.drop(
        columns=[col for col in cols_to_drop if col in train_final.columns]
    )
    test_final = test_final.drop(
        columns=[col for col in cols_to_drop if col in test_final.columns]
    )

    return train_final, test_final

print("Préparation finale des données...")
train_final, test_final = prepare_final_data(train, test, features, metadata)

print(f"Données finales - Train: {train_final.shape}, Test: {test_final.shape}")
print(f"Features disponibles: {len(train_final.columns)}")
print(f"Colonnes: {train_final.columns.tolist()}")

Conversion des features météorologiques...
Préparation finale des données...
Données finales - Train: (15539, 71), Test: (5035, 71)
Features disponibles: 71
Colonnes: ['target', 'temp_mean', 'temp_std', 'temp_min', 'temp_max', 'precip_mean', 'precip_std', 'precip_min', 'precip_max', 'rel_humidity_mean', 'rel_humidity_std', 'rel_humidity_min', 'rel_humidity_max', 'wind_dir_mean', 'wind_dir_std', 'wind_dir_min', 'wind_dir_max', 'wind_spd_mean', 'wind_spd_std', 'wind_spd_min', 'wind_spd_max', 'atmos_press_mean', 'atmos_press_std', 'atmos_press_min', 'atmos_press_max', 'temp_median', 'temp_skew', 'temp_kurt', 'temp_trend', 'temp_roll_mean_10', 'precip_median', 'precip_skew', 'precip_kurt', 'precip_trend', 'precip_roll_mean_10', 'rel_humidity_median', 'rel_humidity_skew', 'rel_humidity_kurt', 'rel_humidity_trend', 'rel_humidity_roll_mean_10', 'wind_dir_median', 'wind_dir_skew', 'wind_dir_kurt', 'wind_dir_trend', 'wind_dir_roll_mean_10', 'wind_spd_median', 'wind_spd_skew', 'wind_spd_kurt', '

In [60]:
# Configuration de la validation croisée
print("Configuration de la validation croisée...")

target_name = "target"
n_splits = 5

# Préparation des features
feature_cols = [col for col in train_final.columns if col not in [target_name, "fold"]]

# Configuration de la validation croisée avec KFold au lieu de TimeSeriesSplit
from sklearn.model_selection import KFold
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

train_final["fold"] = -1

# Marquage des folds
for fold, (train_idx, val_idx) in enumerate(kf.split(train_final)):
    train_final.loc[val_idx, "fold"] = fold

print(f"Validation croisée KFold configurée avec {n_splits} folds")
print(f"Nombre de features: {len(feature_cols)}")

Configuration de la validation croisée...
Validation croisée KFold configurée avec 5 folds
Nombre de features: 70


In [ ]:
# Métriques d'évaluation
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def r2(y_true, y_pred):
    return r2_score(y_true, y_pred)

def evaluate_model(y_true, y_pred):
    """Évalue un modèle avec plusieurs métriques"""
    return {
        "RMSE": rmse(y_true, y_pred),
        "MAE": mae(y_true, y_pred),
        "R2": r2(y_true, y_pred),
    }

# Pour log transform
def evaluate_model_log(y_true, y_pred):
    """Évalue en revenant à l'échelle originale"""
    y_true_orig = np.expm1(y_true)
    y_pred_orig = np.expm1(y_pred)
    return evaluate_model(y_true_orig, y_pred_orig)

print("Métriques d'évaluation définies")

Métriques d'évaluation définies


In [55]:
# Entraînement du modèle LightGBM
def train_lgbm_model(train_data, features, target_name):
    """Entraîne un modèle LightGBM avec validation croisée"""

    oof_predictions = np.zeros(len(train_data))
    scores = []

    for fold in range(5):
        print(f"Fold {fold + 1}/5")

        # Données d'entraînement et de validation
        train_fold = train_data[train_data["fold"] != fold]
        val_fold = train_data[train_data["fold"] == fold]

        X_train = train_fold[features]
        y_train = np.log1p(train_fold[target_name])  # Log transform
        X_val = val_fold[features]
        y_val = np.log1p(val_fold[target_name])

        # Paramètres LightGBM améliorés
        params = {
            "objective": "regression",
            "metric": "rmse",
            "boosting_type": "gbdt",
            "learning_rate": 0.05,
            "max_depth": 10,
            "num_leaves": 50,
            "feature_fraction": 0.8,
            "bagging_fraction": 0.8,
            "bagging_freq": 5,
            "lambda_l1": 0.1,
            "lambda_l2": 0.1,
            "random_state": 42,
            "verbosity": -1,
        }

        # Création des datasets
        train_dataset = lgb.Dataset(X_train, label=y_train)
        val_dataset = lgb.Dataset(X_val, label=y_val, reference=train_dataset)

        # Entraînement
        model = lgb.train(
            params=params,
            train_set=train_dataset,
            valid_sets=[train_dataset, val_dataset],
            num_boost_round=2000,
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
        )

        # Prédictions
        val_pred = model.predict(X_val, num_iteration=model.best_iteration)
        val_pred_orig = np.expm1(val_pred)  # Back to original scale
        oof_predictions[val_fold.index] = val_pred_orig

        # Score
        fold_score = evaluate_model(val_fold[target_name], val_pred_orig)
        scores.append(fold_score)
        print(f"Fold {fold + 1} RMSE: {fold_score['RMSE']:.4f}")
        print(f"Fold {fold + 1} MAE: {fold_score['MAE']:.4f}")
        print(f"Fold {fold + 1} R2: {fold_score['R2']:.4f}")

    overall_score = evaluate_model(train_data[target_name], oof_predictions)
    print(f"Score global RMSE: {overall_score['RMSE']:.4f}")
    print(f"Score global MAE: {overall_score['MAE']:.4f}")
    print(f"Score global R2: {overall_score['R2']:.4f}")

    return oof_predictions, scores, overall_score

print("Entraînement du modèle LightGBM...")
lgbm_oof, lgbm_scores, lgbm_overall_scores = train_lgbm_model(
    train_final, feature_cols, target_name
)

Entraînement du modèle LightGBM...
Fold 1/5
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1996]	training's rmse: 0.0907116	valid_1's rmse: 0.30777
Fold 1 RMSE: 24.7562
Fold 1 MAE: 13.9207
Fold 1 R2: 0.6841
Fold 2/5
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1997]	training's rmse: 0.0894857	valid_1's rmse: 0.322357
Fold 2 RMSE: 25.2816
Fold 2 MAE: 14.2520
Fold 2 R2: 0.6051
Fold 3/5
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1998]	training's rmse: 0.0929352	valid_1's rmse: 0.308373
Fold 3 RMSE: 25.1030
Fold 3 MAE: 13.8632
Fold 3 R2: 0.6401
Fold 4/5
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1995]	training's rmse: 0.0924262	valid_1's rmse: 0.317155
Fold 4 RMSE: 24.7501
Fold 4 MAE: 14.0145
Fold 4 R2: 0.6531
Fold 5/5
Training 

In [56]:
# Entraînement du modèle XGBoost
def train_xgb_model(train_data, features, target_name):
    """Entraîne un modèle XGBoost avec validation croisée"""

    oof_predictions = np.zeros(len(train_data))
    scores = []

    for fold in range(5):
        print(f"Fold {fold + 1}/5")

        train_fold = train_data[train_data["fold"] != fold]
        val_fold = train_data[train_data["fold"] == fold]

        X_train = train_fold[features]
        y_train = np.log1p(train_fold[target_name])
        X_val = val_fold[features]
        y_val = np.log1p(val_fold[target_name])

        # Paramètres XGBoost améliorés
        params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "learning_rate": 0.05,
            "max_depth": 8,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "colsample_bylevel": 0.8,
            "alpha": 0.1,
            "lambda": 0.1,
            "random_state": 42,
            "verbosity": 0,
        }

        # Entraînement
        model = xgb.train(
            params=params,
            dtrain=xgb.DMatrix(X_train, label=y_train),
            num_boost_round=2000,
            evals=[(xgb.DMatrix(X_val, label=y_val), "validation")],
            early_stopping_rounds=100,
            verbose_eval=0,
        )

        # Prédictions
        val_pred = model.predict(
            xgb.DMatrix(X_val), iteration_range=(0, model.best_iteration)
        )
        val_pred_orig = np.expm1(val_pred)
        oof_predictions[val_fold.index] = val_pred_orig

        fold_score = evaluate_model(val_fold[target_name], val_pred_orig)
        scores.append(fold_score)
        print(f"Fold {fold + 1} RMSE: {fold_score['RMSE']:.4f}")
        print(f"Fold {fold + 1} MAE: {fold_score['MAE']:.4f}")
        print(f"Fold {fold + 1} R2: {fold_score['R2']:.4f}")

    overall_score = evaluate_model(train_data[target_name], oof_predictions)
    print(f"Score global RMSE: {overall_score['RMSE']:.4f}")
    print(f"Score global MAE: {overall_score['MAE']:.4f}")
    print(f"Score global R2: {overall_score['R2']:.4f}")

    return oof_predictions, scores, overall_score

print("Entraînement du modèle XGBoost...")
xgb_oof, xgb_scores, xgb_overall_scores = train_xgb_model(
    train_final, feature_cols, target_name
)

Entraînement du modèle XGBoost...
Fold 1/5
Fold 1 RMSE: 24.5902
Fold 1 MAE: 13.6212
Fold 1 R2: 0.6883
Fold 2/5
Fold 2 RMSE: 25.5342
Fold 2 MAE: 14.0683
Fold 2 R2: 0.5971
Fold 3/5
Fold 3 RMSE: 25.3560
Fold 3 MAE: 13.8114
Fold 3 R2: 0.6328
Fold 4/5
Fold 4 RMSE: 24.6092
Fold 4 MAE: 13.7835
Fold 4 R2: 0.6570
Fold 5/5
Fold 5 RMSE: 27.4491
Fold 5 MAE: 14.4382
Fold 5 R2: 0.6035
Score global RMSE: 25.5290
Score global MAE: 13.9445
Score global R2: 0.6370


In [57]:
# Entraînement du modèle CatBoost
def train_catboost_model(train_data, features, target_name):
    """Entraîne un modèle CatBoost avec validation croisée"""

    oof_predictions = np.zeros(len(train_data))
    scores = []

    for fold in range(5):
        print(f"Fold {fold + 1}/5")

        train_fold = train_data[train_data["fold"] != fold]
        val_fold = train_data[train_data["fold"] == fold]

        X_train = train_fold[features]
        y_train = np.log1p(train_fold[target_name])
        X_val = val_fold[features]
        y_val = np.log1p(val_fold[target_name])

        # Paramètres CatBoost améliorés
        model = cat.CatBoostRegressor(
            iterations=2000,
            learning_rate=0.05,
            depth=8,
            l2_leaf_reg=3,
            random_state=42,
            verbose=0,
            early_stopping_rounds=100,
        )

        # Entraînement
        model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

        # Prédictions
        val_pred = model.predict(X_val)
        val_pred_orig = np.expm1(val_pred)
        oof_predictions[val_fold.index] = val_pred_orig

        fold_score = evaluate_model(val_fold[target_name], val_pred_orig)
        scores.append(fold_score)
        print(f"Fold {fold + 1} RMSE: {fold_score['RMSE']:.4f}")
        print(f"Fold {fold + 1} MAE: {fold_score['MAE']:.4f}")
        print(f"Fold {fold + 1} R2: {fold_score['R2']:.4f}")

    overall_score = evaluate_model(train_data[target_name], oof_predictions)
    print(f"Score global RMSE: {overall_score['RMSE']:.4f}")
    print(f"Score global MAE: {overall_score['MAE']:.4f}")
    print(f"Score global R2: {overall_score['R2']:.4f}")

    return oof_predictions, scores, overall_score

print("Entraînement du modèle CatBoost...")
cat_oof, cat_scores, cat_overall_scores = train_catboost_model(
    train_final, feature_cols, target_name
)

Entraînement du modèle CatBoost...
Fold 1/5
Fold 1 RMSE: 25.9060
Fold 1 MAE: 14.5792
Fold 1 R2: 0.6541
Fold 2/5
Fold 2 RMSE: 25.8986
Fold 2 MAE: 14.7134
Fold 2 R2: 0.5856
Fold 3/5
Fold 3 RMSE: 26.3138
Fold 3 MAE: 14.4593
Fold 3 R2: 0.6046
Fold 4/5
Fold 4 RMSE: 25.5514
Fold 4 MAE: 14.5679
Fold 4 R2: 0.6303
Fold 5/5
Fold 5 RMSE: 28.4287
Fold 5 MAE: 15.3714
Fold 5 R2: 0.5747
Score global RMSE: 26.4398
Score global MAE: 14.7382
Score global R2: 0.6106


In [58]:
# Comparaison des modèles
print("Comparaison des modèles:")
print(f"LightGBM RMSE: {lgbm_overall_scores['RMSE']:.4f}, MAE: {lgbm_overall_scores['MAE']:.4f}, R2: {lgbm_overall_scores['R2']:.4f}")
print(f"XGBoost RMSE: {xgb_overall_scores['RMSE']:.4f}, MAE: {xgb_overall_scores['MAE']:.4f}, R2: {xgb_overall_scores['R2']:.4f}")
print(f"CatBoost RMSE: {cat_overall_scores['RMSE']:.4f}, MAE: {cat_overall_scores['MAE']:.4f}, R2: {cat_overall_scores['R2']:.4f}")

# Ensemble
ensemble_oof = (lgbm_oof + xgb_oof + cat_oof) / 3
ensemble_overall_scores = evaluate_model(train_final[target_name], ensemble_oof)
print(f"Ensemble RMSE: {ensemble_overall_scores['RMSE']:.4f}, MAE: {ensemble_overall_scores['MAE']:.4f}, R2: {ensemble_overall_scores['R2']:.4f}")

# Meilleur modèle
models_rmse_scores = {"LightGBM": lgbm_overall_scores['RMSE'], "XGBoost": xgb_overall_scores['RMSE'], "CatBoost": cat_overall_scores['RMSE'], "Ensemble": ensemble_overall_scores['RMSE']}

models_mae_scores = {"LightGBM": lgbm_overall_scores['MAE'], "XGBoost": xgb_overall_scores['MAE'], "CatBoost": cat_overall_scores['MAE'], "Ensemble": ensemble_overall_scores['MAE']}

models_r2_scores = {"LightGBM": lgbm_overall_scores['R2'], "XGBoost": xgb_overall_scores['R2'], "CatBoost": cat_overall_scores['R2'], "Ensemble": ensemble_overall_scores['R2']}

best_model = min(models_rmse_scores, key=models_rmse_scores.get)
print(f"\nMeilleur modèle: {best_model} (RMSE: {models_rmse_scores[best_model]:.4f})")

Comparaison des modèles:
LightGBM RMSE: 25.5298, MAE: 14.1447, R2: 0.6370
XGBoost RMSE: 25.5290, MAE: 13.9445, R2: 0.6370
CatBoost RMSE: 26.4398, MAE: 14.7382, R2: 0.6106
Ensemble RMSE: 25.5271, MAE: 14.0239, R2: 0.6371

Meilleur modèle: Ensemble (RMSE: 25.5271)


In [ ]:
# Entraînement du modèle final sur toutes les données
def train_final_model(train_data, test_data, features, target_name):
    """Entraîne le modèle final sur toutes les données"""

    X_train = train_data[features]
    y_train = train_data[target_name]
    X_test = test_data[features]

    print("Entraînement du modèle final ensemble...")

    # LightGBM
    params_lgb = {
        "objective": "regression",
        "metric": "rmse",
        "boosting_type": "gbdt",
        "learning_rate": 0.05,
        "max_depth": 10,
        "num_leaves": 50,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "lambda_l1": 0.1,
        "lambda_l2": 0.1,
        "random_state": 42,
        "verbosity": -1,
    }

    train_dataset = lgb.Dataset(X_train, label=np.log1p(y_train))
    model_lgb = lgb.train(params=params_lgb, train_set=train_dataset, num_boost_round=2000)
    test_pred_lgb = np.expm1(model_lgb.predict(X_test))

    # XGBoost
    params_xgb = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "learning_rate": 0.05,
        "max_depth": 8,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "colsample_bylevel": 0.8,
        "alpha": 0.1,
        "lambda": 0.1,
        "random_state": 42,
        "verbosity": 0,
    }

    model_xgb = xgb.train(params=params_xgb, dtrain=xgb.DMatrix(X_train, label=np.log1p(y_train)), num_boost_round=2000)
    test_pred_xgb = np.expm1(model_xgb.predict(xgb.DMatrix(X_test)))

    # CatBoost
    model_cat = cat.CatBoostRegressor(iterations=2000, learning_rate=0.05, depth=8, l2_leaf_reg=3, random_state=42, verbose=0)
    model_cat.fit(X_train, np.log1p(y_train))
    test_pred_cat = np.expm1(model_cat.predict(X_test))

    # Ensemble
    final_predictions = (test_pred_lgb + test_pred_xgb + test_pred_cat) / 3

    return final_predictions, {"lgb": model_lgb, "xgb": model_xgb, "cat": model_cat}

# Entraînement du modèle final
final_predictions, final_model = train_final_model(
    train_final, test_final, feature_cols, target_name
)

print("Modèle final entraîné")

Entraînement du modèle final ensemble...
Modèle final entraîné


In [62]:
joblib.dump({"models": final_model, "features": feature_cols}, "pipeline.pkl")
print("Pipeline sauvegardé dans pipeline.pkl")

Pipeline sauvegardé dans pipeline.pkl


# Analyse d'importance des features avec SHAP

In [65]:
# Importance des features
importance = final_model["lgb"].feature_importance(importance_type="gain")
feature_names = final_model["lgb"].feature_name()

# Convertir en DataFrame
importance_df = pd.DataFrame(
    {"feature": feature_names, "importance": importance}
)

# Trier
importance_df = importance_df.sort_values(by="importance", ascending=True)

fig = px.bar(
    importance_df.head(30),
    x="importance",
    y="feature",
    orientation="h",
    title="Top 30 Feature Importance",
)

fig.show()

# Résultats et Prochaines Étapes

## Performance des Modèles
Les trois modèles ont été entraînés avec validation croisée 5-fold KFold:
- **LightGBM**: Excellent pour les données tabulaires avec features numériques
- **XGBoost**: Robuste et performant sur les données structurées
- **CatBoost**: Gère bien les features catégorielles et les valeurs manquantes

Modèle gagnant: Ensemble (XGBoost + LightGBM + CatBoost) avec R2 = 0.64


## Améliorations Apportées
- Feature engineering avancé (moyennes, médianes, skewness, kurtosis, trends, rolling means)
- Intégration des métadonnées des localisations
- Log-transformation de la target
- Changement de TimeSeriesSplit à KFold pour mieux capturer la variance
- Hyperparameter tuning
- Ensembling des modèles

